# Nepali voice pipeline test (prerecorded audio)
Tests local IndicConformer STT → selected Gemma model → Nepali Edge TTS. This is a sequential functional test, not a realtime VAD/browser benchmark.

Before running: enable Internet and T4 x2, add the `HF_TOKEN` Kaggle secret, accept the selected Gemma model terms, and attach datasets containing the IndicConformer assets and a mono 16 kHz PCM WAV file.

In [ ]:
!pip install -q transformers==5.10.1 accelerate==1.12.0 bitsandbytes==0.49.2 sentencepiece==0.2.1 huggingface_hub==0.36.2 onnxruntime-gpu==1.31.1 edge-tts==7.3.1 python-dotenv==1.2.4
import os, sys, gc, time, asyncio, torch
from pathlib import Path
from IPython.display import Audio, display
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
assert torch.cuda.is_available(), 'Enable the Kaggle T4 x2 accelerator.'
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
login(token=HF_TOKEN)
print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Update these two paths after attaching your Kaggle datasets.
INDIC_MODEL_DIR = '/kaggle/input/indicconformer-nepali/indic-conformer-600m-multilingual-onnx'
AUDIO_PATH = '/kaggle/input/nepali-test-audio/sample.wav'
MODEL_NAME = 'gemma4_e2b'  # gemma4_e2b, gemma4_e4b, or gemma4_12b
MODEL_IDS = {
    'gemma4_e2b': 'google/gemma-4-E2B-it',
    'gemma4_e4b': 'google/gemma-4-E4B-it',
    'gemma4_12b': 'google/gemma-4-12B-it',
}
assert Path(INDIC_MODEL_DIR, 'assets').is_dir(), 'Fix INDIC_MODEL_DIR.'
assert Path(AUDIO_PATH).is_file(), 'Fix AUDIO_PATH.'

In [ ]:
# Clone only to reuse the tested STT adapter and prompt definitions.
!if [ -d /kaggle/working/nepali_ai_voice_agent/.git ]; then git -C /kaggle/working/nepali_ai_voice_agent pull -q; else git clone -q --depth 1 https://github.com/BlazeWild/nepali_ai_voice_agent.git /kaggle/working/nepali_ai_voice_agent; fi
sys.path.insert(0, '/kaggle/working/nepali_ai_voice_agent/src')
from nepali_voice_agent.stt import IndicConformer
started = time.perf_counter()
stt = IndicConformer(INDIC_MODEL_DIR)
transcript = stt.transcribe(AUDIO_PATH)
stt_seconds = time.perf_counter() - started
print(f'STT ({stt_seconds:.2f}s): {transcript}')
del stt; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from nepali_voice_agent.prompts import system_prompt
if MODEL_NAME != 'gemma4_e2b' and torch.cuda.device_count() < 2:
    raise RuntimeError('E4B and 12B require T4 x2 for this test.')
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model_id = MODEL_IDS[MODEL_NAME]
limits = {0: '11GiB', 1: '11GiB'} if MODEL_NAME != 'gemma4_e2b' else None
processor = AutoProcessor.from_pretrained(model_id, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(model_id, token=HF_TOKEN, quantization_config=quant, device_map='auto', max_memory=limits, torch_dtype=torch.float16)
mode = 'gemma_e2b' if MODEL_NAME == 'gemma4_e2b' else 'gemma_12b'
messages = [{'role':'system','content':system_prompt(mode)}, {'role':'user','content':transcript}]
rendered = processor.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False, tokenize=False)
inputs = processor(text=rendered, return_tensors='pt').to(model.device)
started = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=300, do_sample=False)
llm_seconds = time.perf_counter() - started
new_tokens = output[0][inputs['input_ids'].shape[-1]:]
answer = processor.decode(new_tokens, skip_special_tokens=True).strip()
print(f'LLM ({llm_seconds:.2f}s): {answer}')
del model, processor, inputs, output; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import edge_tts
TTS_PATH = '/kaggle/working/nepali_response.mp3'
started = time.perf_counter()
await edge_tts.Communicate(answer, 'ne-NP-SagarNeural').save(TTS_PATH)
tts_seconds = time.perf_counter() - started
print({'transcript': transcript, 'answer': answer, 'stt_seconds': round(stt_seconds,2), 'llm_seconds': round(llm_seconds,2), 'tts_seconds': round(tts_seconds,2)})
display(Audio(TTS_PATH))